In [2]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "input.txt")

with open("dataset.txt", 'r', encoding="utf8") as f:
    text = f.read()

with open("input.txt", "r", encoding="utf8") as f:
    input_text = f.read()

In [3]:
f"Length of text dataset: {len(text)}"

'Length of text dataset: 26987'

In [4]:
f"Length of input_text dataset: {len(input_text)}"

'Length of input_text dataset: 1115394'

In [ ]:
print(text[:1000])

In [30]:
chars = list(set(text))
vocab_size = len(chars)
print("vocab size:", vocab_size)
print("".join(chars))


vocab size: 84
🙁Dhf*oEX0mVp-OJu”6🙂t😉A?xU8
″F(PIkjbG:'Sgsz—"ywK.ieaB>)<!H/9q;l5Tn&’r,“N dCY…WvM12LcR


In [7]:
char2id = { ch : idx for idx, ch in enumerate(chars) }
id2char = { idx : ch for idx, ch in enumerate(chars) }

print(char2id)
print("\n\n\n",id2char)

{'🙁': 0, 'D': 1, 'h': 2, 'f': 3, '*': 4, 'o': 5, 'E': 6, 'X': 7, '0': 8, 'm': 9, 'V': 10, 'p': 11, '-': 12, 'O': 13, 'J': 14, 'u': 15, '”': 16, '6': 17, '🙂': 18, 't': 19, '😉': 20, 'A': 21, '?': 22, 'x': 23, 'U': 24, '8': 25, '\n': 26, '″': 27, 'F': 28, '(': 29, 'P': 30, 'I': 31, 'k': 32, 'j': 33, 'b': 34, 'G': 35, ':': 36, "'": 37, 'S': 38, 'g': 39, 's': 40, 'z': 41, '—': 42, '"': 43, 'y': 44, 'w': 45, 'K': 46, '.': 47, 'i': 48, 'e': 49, 'a': 50, 'B': 51, '>': 52, ')': 53, '<': 54, '!': 55, 'H': 56, '/': 57, '9': 58, 'q': 59, ';': 60, 'l': 61, '5': 62, 'T': 63, 'n': 64, '&': 65, '’': 66, 'r': 67, ',': 68, '“': 69, 'N': 70, ' ': 71, 'd': 72, 'C': 73, 'Y': 74, '…': 75, 'W': 76, 'v': 77, 'M': 78, '1': 79, '2': 80, 'L': 81, 'c': 82, 'R': 83}



 {0: '🙁', 1: 'D', 2: 'h', 3: 'f', 4: '*', 5: 'o', 6: 'E', 7: 'X', 8: '0', 9: 'm', 10: 'V', 11: 'p', 12: '-', 13: 'O', 14: 'J', 15: 'u', 16: '”', 17: '6', 18: '🙂', 19: 't', 20: '😉', 21: 'A', 22: '?', 23: 'x', 24: 'U', 25: '8', 26: '\n', 27: '″', 28: 

In [23]:
def encode(text):
    return [ char2id[char] if char in char2id else "[UNK]" for char in text]

def decode(ids):
    return "".join([ id2char[id] if id in id2char else "[UNK]" for id in ids ])

encoded = encode("text")
print(encoded)

decoded = decode(encoded)
print(decoded)

[19, 49, 23, 19]
text


In [52]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.set_default_device(device)


data = torch.tensor(encode(text), dtype = torch.long)

In [53]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [55]:
block_size = 8

x = train_data[:block_size]
print("x: ", x, len(x))
y = train_data[1: block_size + 1]
print("\ny: ", y, len(y))

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(context.data, "target: ", target.data)


x:  tensor([56, 49, 36, 71, 31, 71, 45, 50], device='mps:0') 8

y:  tensor([49, 36, 71, 31, 71, 45, 50, 64], device='mps:0') 8
tensor([56], device='mps:0') target:  tensor(49, device='mps:0')
tensor([56, 49], device='mps:0') target:  tensor(36, device='mps:0')
tensor([56, 49, 36], device='mps:0') target:  tensor(71, device='mps:0')
tensor([56, 49, 36, 71], device='mps:0') target:  tensor(31, device='mps:0')
tensor([56, 49, 36, 71, 31], device='mps:0') target:  tensor(71, device='mps:0')
tensor([56, 49, 36, 71, 31, 71], device='mps:0') target:  tensor(45, device='mps:0')
tensor([56, 49, 36, 71, 31, 71, 45], device='mps:0') target:  tensor(50, device='mps:0')
tensor([56, 49, 36, 71, 31, 71, 45, 50], device='mps:0') target:  tensor(64, device='mps:0')


In [85]:
torch.manual_seed(42)
batch_size = 4

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : block_size + i] for i in ix])
    y = torch.stack([data[i + 1 : block_size + 1 + i] for i in ix])
    return x,y

xb, yb = get_batch("train")
print("shape: ", xb.shape)
print("inputs: \n", xb);
print('\n\n')
print("shape: ",yb.shape)
print("targets: \n", yb)



shape:  torch.Size([4, 8])
inputs: 
 tensor([[45, 48, 40,  2, 71, 31, 71, 45],
        [71,  2, 49, 67, 49, 71, 31, 71],
        [38,  2, 49, 36, 71, 13, 32, 50],
        [45, 47, 26, 38,  2, 49, 36, 71]], device='mps:0')



shape:  torch.Size([4, 8])
targets: 
 tensor([[48, 40,  2, 71, 31, 71, 45, 50],
        [ 2, 49, 67, 49, 71, 31, 71, 32],
        [ 2, 49, 36, 71, 13, 32, 50, 44],
        [47, 26, 38,  2, 49, 36, 71, 21]], device='mps:0')


In [89]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print("input:", context,"  target: ", target)

input: tensor([45], device='mps:0')   target:  tensor(48, device='mps:0')
input: tensor([45, 48], device='mps:0')   target:  tensor(40, device='mps:0')
input: tensor([45, 48, 40], device='mps:0')   target:  tensor(2, device='mps:0')
input: tensor([45, 48, 40,  2], device='mps:0')   target:  tensor(71, device='mps:0')
input: tensor([45, 48, 40,  2, 71], device='mps:0')   target:  tensor(31, device='mps:0')
input: tensor([45, 48, 40,  2, 71, 31], device='mps:0')   target:  tensor(71, device='mps:0')
input: tensor([45, 48, 40,  2, 71, 31, 71], device='mps:0')   target:  tensor(45, device='mps:0')
input: tensor([45, 48, 40,  2, 71, 31, 71, 45], device='mps:0')   target:  tensor(50, device='mps:0')
input: tensor([71], device='mps:0')   target:  tensor(2, device='mps:0')
input: tensor([71,  2], device='mps:0')   target:  tensor(49, device='mps:0')
input: tensor([71,  2, 49], device='mps:0')   target:  tensor(67, device='mps:0')
input: tensor([71,  2, 49, 67], device='mps:0')   target:  tenso